# 07 - Held-out test evaluation: image and fusion models

Generate predictions for **every saved CNN run** and its **NN and spline fusion**
models. Report no metadata-only models. Use `data/test_labels.csv` and `data/test/`,
require disjoint development/test patients, and evaluate all models on exactly
the same test images. No model fitting, threshold tuning or model selection occurs here.

The image inference policy is the **mean probability of the three saved CNN folds**.
Each final all-development logistic stacker consumes that mean and the corresponding
metadata test score. This differs from single-fold OOF inputs used for development;
calibration/threshold transfer is unvalidated and test recall may miss its nominal target.

95% confidence intervals resample whole patients, carrying all their images and
allowing repeated sampled patients. Metrics remain image-level. Thresholds are
held fixed in every bootstrap sample. These intervals measure test sampling
uncertainty conditional on fitted models, not training or model-selection uncertainty.

**Run order:** 06 fusion, 05 thresholds, then this notebook. If metadata test inputs
are absent, save image-only results as an explicitly **partial** report; never
substitute development OOF metadata or silently omit failed fusion models.

In [1]:
import sys, json, hashlib, gc
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay
from melanoma_detection.utils import RANDOM_STATE
from melanoma_detection.final_evaluation import (
    validate_test_cohort, align_test_predictions, patient_bootstrap_report)
from melanoma_detection.fusion_model import predict_fusion_model

RESULTS = ROOT / "results"
TEST_LABELS_PATH = ROOT / "data/test_labels.csv"
TEST_IMAGE_DIR = ROOT / "data/test"
METADATA_TEST_PATH = RESULTS / "predictions/metadata_nn_test_d1.csv"
METADATA_DEVELOPMENT_PATH = RESULTS / "predictions/metadata_nn_oof_d1.csv"
MANIFEST_PATH = RESULTS / "metrics/fusion_all_models_manifest.json"
THRESHOLDS_PATH = RESULTS / "thresholds/pooled_threshold_comparison.csv"
OUTPUT_DIR = RESULTS / "final_evaluation"
PREDICTION_DIR = OUTPUT_DIR / "predictions"
N_BOOTSTRAP = 1000
CONFIDENCE = 0.95
BATCH_SIZE = 8
NUM_WORKERS = 0
REUSE_PREDICTIONS = True
POLICIES = ["fixed_0.5", "recall_0.9", "recall_0.95"]
# Recall policies are frozen development OOF candidates, not test-calibrated guarantees.

## 1. Validate the held-out cohort and freeze operating points

Check test patients against both image and metadata development populations and
against every saved CNN split file. Reject duplicate/missing IDs, invalid labels,
missing JPEGs and development overlap. Paths and thresholds are resolved before
prediction. All runs in the development manifest are included.

In [2]:
for path in (TEST_LABELS_PATH, METADATA_DEVELOPMENT_PATH, MANIFEST_PATH, THRESHOLDS_PATH):
    if not path.is_file(): raise FileNotFoundError(f"Required input: {path}")
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
test_rows = pd.read_csv(TEST_LABELS_PATH)
metadata_development = pd.read_csv(METADATA_DEVELOPMENT_PATH)
validate_test_cohort(test_rows, metadata_development)
if "split" in test_rows and not test_rows.split.eq("test").all():
    raise ValueError("The supplied test table contains non-test rows.")
test_rows["file_path"] = test_rows.image_name.map(lambda name: str(TEST_IMAGE_DIR / f"{name}.jpg"))
missing_images = [path for path in test_rows.file_path if not Path(path).is_file()]
if missing_images: raise FileNotFoundError(f"Missing test images: {missing_images[:5]}")
checkpoint_paths = {}
for run, oof_path in manifest["image_oof_paths"].items():
    development = pd.read_csv(oof_path)
    validate_test_cohort(test_rows, development)
    checkpoint_paths[run] = []
    for fold in (1, 2, 3):
        folder = Path(oof_path).parent / f"fold_{fold}" / run
        checkpoint = folder / "model.pt"
        if not checkpoint.is_file(): raise FileNotFoundError(checkpoint)
        saved_split = pd.read_csv(folder / "splits.csv")
        if set(test_rows.patient_id) & set(saved_split.patient_id):
            raise ValueError(f"Test patient found in checkpoint development split: {checkpoint}")
        checkpoint_paths[run].append(checkpoint)
    for variant in ("NN", "Spline"):
        path = RESULTS / "models/fusion" / run / f"fusion_image_metadata_{variant.lower()}.joblib"
        if not path.is_file(): raise FileNotFoundError(path)
threshold_table = pd.read_csv(THRESHOLDS_PATH)
frozen_thresholds = {}
for run in checkpoint_paths:
    for variant in ("CNN only", "CNN + NN", "CNN + Spline"):
        selected = threshold_table.loc[threshold_table.image_run.eq(run) & threshold_table.variant.eq(variant)]
        if selected.policy.duplicated().any() or not set(POLICIES).issubset(selected.policy):
            raise ValueError(f"Missing/duplicate development threshold policies: {run} / {variant}")
        values = selected.set_index("policy").loc[POLICIES, "threshold"]
        if not np.isfinite(values).all() or not values.between(0, 1).all():
            raise ValueError("Invalid frozen thresholds.")
        frozen_thresholds[run, variant] = values.to_dict()
print(f"Held-out cohort: {len(test_rows)} images; {test_rows.patient_id.nunique()} patients.")
print(f"Planned models: {3 * len(checkpoint_paths)} (image and fusion only).")
for folder in (OUTPUT_DIR, PREDICTION_DIR, OUTPUT_DIR / "confusion_matrices"):
    folder.mkdir(parents=True, exist_ok=True)
threshold_table.to_csv(OUTPUT_DIR / "frozen_development_thresholds.csv", index=False)

Held-out cohort: 1988 images; 106 patients.
Planned models: 24 (image and fusion only).


## 2. Generate image test predictions from saved checkpoints

Load saved preprocessing and weights with no pretrained download or retraining.
Use GPU when available, otherwise CPU. Load one fold at a time, average its
probabilities with the other two folds, then release it. Cache provenance includes
checkpoint fingerprints, test labels and image file signatures; mismatches raise
instead of reusing stale predictions. Set `REUSE_PREDICTIONS=False` to regenerate
predictions from the same frozen models; do not use test performance to retune.

In [ ]:
import torch
from torch.utils.data import DataLoader
from melanoma_detection.data import MelanomaImageDataset
from melanoma_detection.image_model import load_saved_image_model, predict_image_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Inference device:", device)

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

test_signature = {"labels_sha256": file_hash(TEST_LABELS_PATH),
    "images": [{"path": path, "size": Path(path).stat().st_size,
                "mtime_ns": Path(path).stat().st_mtime_ns} for path in test_rows.file_path]}
test_scores, model_details = {}, {}
for run, paths in checkpoint_paths.items():
    cache = PREDICTION_DIR / f"{run}__image.csv"
    provenance_path = cache.with_suffix(".json")
    provenance = {"checkpoints": {str(path): file_hash(path) for path in paths},
                  "test": test_signature, "aggregation": "mean of three fold probabilities"}
    if REUSE_PREDICTIONS and cache.is_file():
        if not provenance_path.is_file() or json.loads(provenance_path.read_text()) != provenance:
            raise ValueError(f"Stale/unverifiable prediction cache: {cache}")
        table = align_test_predictions(pd.read_csv(cache), test_rows, ["score"])
    else:
        fold_scores = []
        for fold, path in enumerate(paths, 1):
            model, transform, metadata = load_saved_image_model(path, device=device)
            dataset = MelanomaImageDataset(test_rows, transform)
            loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")
            raw = predict_image_model(model, loader, device)
            raw = align_test_predictions(raw, test_rows, ["melanoma_score"])
            fold_scores.append(raw.melanoma_score.to_numpy())
            del model, loader, dataset
            gc.collect()
            if device.type == "cuda": torch.cuda.empty_cache()
            print(f"{run}: predicted fold {fold}", flush=True)
        table = test_rows[["image_name", "patient_id", "target"]].copy()
        for fold, scores in enumerate(fold_scores, 1): table[f"fold_{fold}_score"] = scores
        table["score"] = np.mean(fold_scores, axis=0)
        table = align_test_predictions(table, test_rows, ["score"])
        table.to_csv(cache, index=False)
        provenance_path.write_text(json.dumps(provenance, indent=2), encoding="utf-8")
    name = f"{run} | CNN only"
    test_scores[name] = table
    model_details[name] = (run, "CNN only")
print("Saved image predictions for all CNN runs.")

## 3. Generate fusion test predictions

No metadata-only results are reported. Fusion nevertheless requires two test
inputs: the image score and metadata score. `METADATA_TEST_PATH` must contain
`image_name`, `patient_id`, `p_nn_mean` and `p_spline` for every test image;
`target` is optional and is checked if present. These probabilities must come
from metadata models/preprocessing fitted only on development patients.
The development OOF file is not a replacement for test probabilities.

Apply each saved final logistic stacker to the CNN fold-mean score and its
metadata input. Missing metadata test inputs produce an explicit partial report.
Invalid or incomplete supplied inputs raise errors rather than dropping rows.

In [ ]:
missing_models = []
if not METADATA_TEST_PATH.is_file():
    missing_models = [f"{run} | CNN + {variant}" for run in checkpoint_paths for variant in ("NN", "Spline")]
    print("PARTIAL REPORT: fusion test inputs are missing:", METADATA_TEST_PATH)
    print(f"{len(missing_models)} fusion models cannot be evaluated; no substitute scores were generated.")
else:
    metadata_test = pd.read_csv(METADATA_TEST_PATH)
    if "patient_id" not in metadata_test: raise ValueError("Metadata test scores require patient_id.")
    metadata_test = align_test_predictions(metadata_test, test_rows, ["p_nn_mean", "p_spline"])
    for run in checkpoint_paths:
        inputs = test_rows[["image_name", "patient_id", "target"]].copy()
        inputs["image_score"] = test_scores[f"{run} | CNN only"].score.to_numpy()
        inputs["metadata_score"] = metadata_test.p_nn_mean.to_numpy()
        inputs["p_spline"] = metadata_test.p_spline.to_numpy()
        for variant in ("NN", "Spline"):
            path = RESULTS / "models/fusion" / run / f"fusion_image_metadata_{variant.lower()}.joblib"
            model = joblib.load(path)
            output = inputs.assign(score=predict_fusion_model(model, inputs))
            output = align_test_predictions(output, test_rows, ["score"])
            name = f"{run} | CNN + {variant}"
            test_scores[name] = output
            model_details[name] = (run, f"CNN + {variant}")
            output.to_csv(PREDICTION_DIR / f"{run}__fusion_{variant.lower()}.csv", index=False)
            (PREDICTION_DIR / f"{run}__fusion_{variant.lower()}.json").write_text(json.dumps({
                "fusion_checkpoint": str(path), "checkpoint_sha256": file_hash(path),
                "metadata_test_sha256": file_hash(METADATA_TEST_PATH),
                "image_predictions_sha256": file_hash(PREDICTION_DIR / f"{run}__image.csv"),
                "metadata_training_provenance": "Must be verified by the producer; no fitting occurs here."
            }, indent=2), encoding="utf-8")
print(f"Available: {len(test_scores)} / {3 * len(checkpoint_paths)} planned models.")

## 4. Test metrics with patient-level 95% confidence intervals

Report ROC-AUC, AP, sensitivity, specificity, balanced accuracy, precision, F1 and
TN/FP/FN/TP. Patient-cluster percentile intervals accompany the seven performance
metrics. Resamples lacking both classes are skipped; undefined metric values
are excluded per metric and valid counts are recorded. Do not interpret intervals
as proof of clinical significance or use these results to choose new thresholds.

In [ ]:
reports = []
for name, table in test_scores.items():
    run, variant = model_details[name]
    report = patient_bootstrap_report(table, frozen_thresholds[run, variant],
        n_boot=N_BOOTSTRAP, seed=RANDOM_STATE, confidence=CONFIDENCE)
    report["model"], report["image_run"], report["variant"] = name, run, variant
    report["threshold_source"] = "frozen development OOF; transfer to final pipeline unvalidated"
    reports.append(report)
    print("Completed patient bootstrap:", name, flush=True)
metrics = pd.concat(reports, ignore_index=True)
metrics["evaluation_complete"] = not missing_models
metrics.to_csv(OUTPUT_DIR / "test_metrics_patient_ci.csv", index=False)
for policy in POLICIES:
    table = metrics.loc[metrics.policy.eq(policy)].copy()
    for metric in ("ROC_AUC", "AP", "Sensitivity", "Specificity", "Balanced accuracy"):
        table[f"{metric} 95% CI"] = table.apply(
            lambda row: f"[{row[metric + '_lower']:.3f}, {row[metric + '_upper']:.3f}]", axis=1)
    print(policy, "(PARTIAL: image only)" if missing_models else "")
    display(table[["model", "threshold", "ROC_AUC", "ROC_AUC 95% CI", "AP", "AP 95% CI",
                   "Sensitivity", "Sensitivity 95% CI", "Specificity", "Specificity 95% CI",
                   "Balanced accuracy", "TN", "FP", "FN", "TP"]].round(4))

## 5. Confusion matrices and evaluation manifest

One figure per CNN run: rows are operating-point policies, columns are image-only
and available fusion variants. Counts use the same frozen thresholds as the
metrics table. Save all plots and a completeness manifest. No metadata-only
models appear, and no test-set winner is selected.

In [ ]:
for run in checkpoint_paths:
    names = [name for name, detail in model_details.items() if detail[0] == run]
    fig, axes = plt.subplots(len(POLICIES), len(names), figsize=(4.5 * len(names), 10), squeeze=False)
    for i, policy in enumerate(POLICIES):
        for j, name in enumerate(names):
            row = metrics.loc[metrics.model.eq(name) & metrics.policy.eq(policy)].iloc[0]
            cm = np.array([[row.TN, row.FP], [row.FN, row.TP]], dtype=int)
            ConfusionMatrixDisplay(cm, display_labels=["Non-melanoma", "Melanoma"]).plot(
                ax=axes[i,j], values_format="d", colorbar=False, cmap="Blues")
            axes[i,j].set_title(f"{model_details[name][1]} | {policy}\nthreshold={row.threshold:.5f}")
    fig.suptitle(run + (" (PARTIAL: image only)" if missing_models else ""))
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "confusion_matrices" / f"{run}.png", dpi=150, bbox_inches="tight")
    plt.show()
(OUTPUT_DIR / "evaluation_manifest.json").write_text(json.dumps({
    "complete": not missing_models, "evaluated_models": list(test_scores), "missing_models": missing_models,
    "test_labels": str(TEST_LABELS_PATH), "test_labels_sha256": file_hash(TEST_LABELS_PATH),
    "test_images": str(TEST_IMAGE_DIR), "metadata_test_path": str(METADATA_TEST_PATH),
    "thresholds_sha256": file_hash(THRESHOLDS_PATH), "thresholds_source": str(THRESHOLDS_PATH),
    "inference_policy": "mean CNN fold probabilities; final all-development fusion LR",
    "bootstrap_unit": "patient", "bootstrap_repetitions": N_BOOTSTRAP, "seed": RANDOM_STATE,
    "confidence": CONFIDENCE, "threshold_transfer_validated": False
}, indent=2), encoding="utf-8")
print("Saved report:", OUTPUT_DIR)
print("PARTIAL: missing fusion test inputs." if missing_models else "Complete image and fusion test evaluation.")